# 05 – Sitting vs standing

Both baselines (RF, 1D-CNN) confuse sitting and standing most. With a waist phone the two postures look almost identical: no motion, and the only difference is a small tilt of the hip.

Tried here:
- **(a) gravity angles**: angle between gravity (`total_acc - body_acc`) and each phone axis, fed to the CNN either as 3 extra channels or as window means concatenated before the dense layer.
- **(b) 5-class "stationary"**: sitting+standing merged, scored from the 6-class predictions.

Each variant is trained with 3 seeds; differences under ~1 point are noise.

In [1]:
import os, sys, yaml, numpy as np
sys.path.insert(0, "../src")
cfg = yaml.safe_load(open("../config/config.yaml"))
ROOT = os.environ.get("UCI_ROOT", "../data/uci/UCI HAR Dataset")
FIG = "../reports/figures"
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from data_loader import load_uci, split_val
from preprocess import normalize, gravity_angles
from models import build_cnn, build_cnn_gravity
from train import fit
from evaluate import report
d = load_uci(ROOT)
tr, va, te = *split_val(d["train"], cfg["split"]["uci_val"]), d["test"]
ns = "../" + cfg["norm_stats"]
X = [normalize(s["X_raw"], ns) for s in (tr, va, te)]
A = [gravity_angles(s["X_raw"]) for s in (tr, va, te)]           # [N,128,3], radians
mu, sd = A[0].mean((0, 1)), A[0].std((0, 1))                      # train-only stats
A = [(a - mu) / sd for a in A]
Am = [a.mean(1) for a in A]                                       # [N,3] window means
y = [s["y"] for s in (tr, va, te)]

## Why it is hard: gravity angle distributions (test set)

In [2]:
import matplotlib.pyplot as plt
raw = gravity_angles(te["X_raw"]).mean(1)
fig, axs = plt.subplots(1, 3, figsize=(12, 3))
for i, ax in enumerate(axs):
    for c, n in [(3, "sitting"), (4, "standing")]:
        ax.hist(np.degrees(raw[te["y"] == c, i]), bins=40, alpha=.6, label=n)
    ax.set_title(f"angle to axis {'xyz'[i]} (deg)"); ax.legend()
fig.tight_layout()

In [3]:
def run(name, build, xs, seeds=(42, 43, 44)):
    rows, preds = [], []
    for s in seeds:
        m, _ = fit(build(), xs[0], y[0], xs[1], y[1], {**cfg, "seed": s})
        p = m.predict(xs[2], verbose=0).argmax(1)
        p5, y5 = np.where(p == 4, 3, p), np.where(y[2] == 4, 3, y[2])   # standing -> sitting = "stationary"
        cm = confusion_matrix(y[2], p, labels=range(6)); sit_stand = cm[3, 4] + cm[4, 3]
        rows.append([accuracy_score(y[2], p), f1_score(y[2], p, average="macro"),
                     accuracy_score(y5, p5), f1_score(y5, p5, average="macro"), sit_stand])
        preds.append(p)
    r = np.array(rows)
    print(f"{name:28s} acc6 {r[:,0].mean():.3f}±{r[:,0].std():.3f}  F1_6 {r[:,1].mean():.3f}  "
          f"acc5 {r[:,2].mean():.3f}  F1_5 {r[:,3].mean():.3f}  sit<->stand errors {r[:,4].mean():.0f}")
    return r, preds

res = {}
res["baseline CNN (9 ch)"] = run("baseline CNN (9 ch)", build_cnn, X)
res["CNN + angle channels (12 ch)"] = run("CNN + angle channels (12 ch)", lambda: build_cnn((128, 12)),
                                          [np.concatenate([x, a], -1) for x, a in zip(X, A)])
res["CNN + angle dense branch"] = run("CNN + angle dense branch", build_cnn_gravity,
                                      [[x, a] for x, a in zip(X, Am)])

Epoch 1/30


99/99 - 5s - 46ms/step - accuracy: 0.8378 - loss: 0.4180 - val_accuracy: 0.9412 - val_loss: 0.1776


Epoch 2/30


99/99 - 2s - 22ms/step - accuracy: 0.9486 - loss: 0.1151 - val_accuracy: 0.9449 - val_loss: 0.1715


Epoch 3/30


99/99 - 2s - 19ms/step - accuracy: 0.9565 - loss: 0.0933 - val_accuracy: 0.9552 - val_loss: 0.1360


... (214 outputs left out)

99/99 - 2s - 20ms/step - accuracy: 0.9798 - loss: 0.0495 - val_accuracy: 0.9477 - val_loss: 0.1651


Epoch 12/30


99/99 - 2s - 22ms/step - accuracy: 0.9761 - loss: 0.0618 - val_accuracy: 0.9197 - val_loss: 0.3131


CNN + angle dense branch     acc6 0.919±0.006  F1_6 0.919  acc5 0.977  F1_5 0.972  sit<->stand errors 171


## Best variant: confusion matrix (best seed by 6-class macro F1)

In [4]:
best = max(res, key=lambda k: res[k][0][:, 1].mean())
i = res[best][0][:, 1].argmax()
print(best, "seed", (42, 43, 44)[i])
report(y[2], res[best][1][i], FIG, "sitstand_best")

CNN + angle dense branch seed 44


{'accuracy': 0.9246691550729556,
 'macro_f1': 0.9241325316944543,
 'f1_per_activity': {'WALKING': np.float64(0.978),
  'WALKING_UPSTAIRS': np.float64(0.9422),
  'WALKING_DOWNSTAIRS': np.float64(0.9436),
  'SITTING': np.float64(0.8193),
  'STANDING': np.float64(0.8617),
  'LAYING': np.float64(1.0)}}

## Conclusion

- No variant reduces sit<->stand errors: about 160 of 1,023 test sit/stand windows stay confused in every variant.
- The dense-branch gain (+0.5 pt mean) is in other classes and within seed noise. TF on CPU is not bit-deterministic, and the baseline moved 0.5 pt between two identical runs.
- Why: the gravity-angle histograms above overlap. A waist phone tilts by only a few degrees between sitting and standing, and that varies more between people than between the two postures. `total_acc` already carries the tilt, so the angle features add almost nothing new.
- If the app can live with one 'stationary' label, accuracy goes from ~0.91 to ~0.97 (5 classes).
